## ⚡ Optimización de Delta Tables

Hasta ahora hemos explorado cómo una Delta Table almacena información mediante dos componentes fundamentales:

* 📦 Archivos de datos (Parquet)
* 📝 Transaction Log (_delta_log)

Sin embargo, a medida que una tabla recibe operaciones como:

* INSERT
* UPDATE
* DELETE
* MERGE
* Streaming Writes

es común que se generen una gran cantidad de archivos pequeños (*small files*).

---

### 🤔 ¿Por qué esto representa un problema?

Aunque Spark es un motor distribuido, leer miles de archivos pequeños genera un overhead considerable. Y, antes de procesar los datos, Spark debe:

* 📂 Localizar cada archivo
* 📖 Abrir cada archivo
* 📋 Leer sus metadatos
* 🔄 Coordinar múltiples tareas de lectura

Por esta razón, suele ser más eficiente leer:

* ✅ 1 archivo de 1 GB que ❌ 1000 archivos de 1 MB

No porque el volumen de datos cambie, sino porque disminuye significativamente el costo asociado a la gestión de archivos.

---

###  Formas de optimización:

* 🚀 OPTIMIZE
* 🎯 OPTIMIZE + ZORDER

---
### ⚠️ ¿Cuándo deberíamos ejecutar OPTIMIZE?

Un error común es pensar que debemos optimizar una tabla cuando acumula muchas versiones en su historial. Pero, la realidad es distinta...

El criterio principal no es la cantidad de versiones registradas, sino la presencia de **small files**.

* #### Caso 1

    * 📜 1000 versiones
    * 📦 10 archivos Parquet bien dimensionados
    * ➡️ El beneficio de ejecutar OPTIMIZE sería reducido.

* #### Caso 2

    * 📜 50 versiones
    * 📦 Miles de archivos pequeños generados por INSERT, UPDATE, DELETE o MERGE
    * ➡️ Ejecutar OPTIMIZE puede mejorar significativamente el rendimiento de las consultas.



### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("04OptimizingTables").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ Utilizaremos las siguientes Delta Table para las demostraciones:

- Tabla 1: `demo_optimize`
- Tabla 2: `demo_zorder`



In [0]:
## TABLA PARA UTILIZAR EN EL EJEMPLO DE OPTIMIZE:
spark.sql("""
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.demo_optimize
        (
            id INT,
            nombre STRING
        )
        USING DELTA;
          """)
print("Tabla creada correctamente")

## REGISTRAR DATOS PARA LA TABLA DEL EJEMPLO DE OPTIMIZE:
for i in range(100):

    spark.sql(f"""
    INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.demo_optimize
    VALUES ({i}, 'Cliente_{i}')
    """)
print("Datos registrados correctamente")

In [0]:
## TABLA PARA UTILIZAR EN EL EJEMPLO DE OPTIMIZE + ZORDER:
spark.sql("""
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.demo_zorder
        (
            id_cliente INT,
            ciudad STRING,
            monto DOUBLE
        )
        USING DELTA;          
          """)
print("Tabla creada correctamente")


## REGISTRAR DATOS PARA LA TABLA DEL EJEMPLO DE OPTIMIZE + ZORDER:
# # from random import choice
# # from random import randint

# # ciudades = [
# #     "Lima",
# #     "Bogota",
# #     "Quito",
# #     "Santiago"
# # ]

# # for i in range(10000):

# #     ciudad = choice(ciudades)

# #     spark.sql(f"""
# #     INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.demo_zorder
# #     VALUES
# #     (
# #         {i},
# #         '{ciudad}',
# #         {randint(100,1000)}
# #     )
# #     """)
## 💡 Estos registros mediante el for puede tomar tiempo (No ejecutar en caso tengas demasiado tiempo libre).
# print("Datos registrados correctamente") ## Tiempo de registro de datos: 3 horas con 13 min.

### 🚀 OPTIMIZE

El comando `OPTIMIZE` permite compactar múltiples archivos pequeños en archivos más grandes y eficientes para la lectura analítica.

```sql
OPTIMIZE catalog.schema.table;
```

Al ejecutarlo:

* ✅ Se compactan archivos Parquet
* ✅ Se genera una nueva versión en el historial de la tabla
* ✅ Los archivos anteriores quedan marcados como obsoletos
* ✅ Se mejora el rendimiento de lectura


In [0]:
## VERIFICAR CANTIDAD DE ARCHIVOS (PARQUET)
display(spark.sql("DESCRIBE DETAIL catalog_databricks_2026_de.schema_databricks_2026_de.demo_optimize"))
### Resultado: numFiles ➡️ 7

## APLICAR OPTIMIZE PARA REDUCIR CANTIDAD DE ARCHIVOS PEQUEÑOS
spark.sql("OPTIMIZE catalog_databricks_2026_de.schema_databricks_2026_de.demo_optimize")

## VERIFICAR CANTIDAD DE ARCHIVOS (PARQUET)
display(spark.sql("DESCRIBE DETAIL catalog_databricks_2026_de.schema_databricks_2026_de.demo_optimize"))
### Resultado: numFiles ➡️ 1

## 💡 Se logró reducir los archivos pequeños y compactar en 1 solo. En escalas mas grandes, la diferencia es notoria.


### 🎯 OPTIMIZE + ZORDER

En algunos escenarios no solo queremos compactar archivos, sino también optimizar la forma en que los datos quedan organizados físicamente. Para ello podemos utilizar:

```sql
OPTIMIZE catalog.schema.table
ZORDER BY (columna1, columna2, columna3);
```

Además de compactar archivos, Delta Lake reorganiza físicamente los datos utilizando las columnas especificadas. Esto permite que Spark reduzca la cantidad de datos que necesita leer cuando existen filtros sobre dichas columnas.

Beneficios:

* ✅ Menor cantidad de archivos escaneados
* ✅ Menor volumen de datos leídos
* ✅ Consultas más rápidas
* ✅ Mejor aprovechamiento del Data Skipping

> 💡 ZORDER resulta especialmente útil sobre columnas utilizadas frecuentemente en filtros (`WHERE`), búsquedas o consultas analíticas recurrentes.

In [0]:
## VERIFICAR CANTIDAD DE ARCHIVOS (PARQUET)
display(spark.sql("DESCRIBE DETAIL catalog_databricks_2026_de.schema_databricks_2026_de.demo_zorder"))
### Resultado: numFiles ➡️ 18

## APLICAR OPTIMIZE PARA REDUCIR CANTIDAD DE ARCHIVOS PEQUEÑOS Y ORDENARLOS POR CIUDAD (LECTURA EFICIENTE)
spark.sql("""
          OPTIMIZE catalog_databricks_2026_de.schema_databricks_2026_de.demo_zorder
          ZORDER BY (ciudad)
          """)

## VERIFICAR CANTIDAD DE ARCHIVOS (PARQUET)
display(spark.sql("DESCRIBE DETAIL catalog_databricks_2026_de.schema_databricks_2026_de.demo_zorder"))
### Resultado: numFiles ➡️ 1

## 💡 Se logró reducir los archivos pequeños y compactar en 1 solo. Además, se mejoró la eficiencia en la lectura
##     debido a que Spark al filtrar por ciudad, no va a escanear varios archivos. En escalas mas grandes, la diferencia es notoria.

### 🎓 A tener en cuenta

* `OPTIMIZE` no existe para reducir versiones del historial ni para limpiar el Transaction Log. Sino, es combatir el problema de los **small files**, compactando archivos Parquet para que Spark pueda procesar la información de manera más eficiente.

* En cargas frecuentes o entornos productivos, una estrategia adecuada de optimización puede marcar una diferencia importante en el rendimiento de las consultas analíticas.
